# Bronze → Silver: Account Domain — CashTransaction.txt
## silver.cash_transactions (CREATE OR REPLACE, full rebuild)

**Source:** `bronze.cashtransaction` (B1=1,203,664 + B2=637 + B3=642 = 1,204,943 total)

**Processing Pattern:** Full Rebuild → CREATE OR REPLACE with dedup
- Not incremental append — Silver is full rebuild from all Bronze each run
- CDC_FLAG = 'I' only (all inserts, no updates or deletes)

**Silver Layer Contract (per MD):**
- DROPPED: `_ingest_ts` (bronze-only), `_source_file` (no longer needed)
- ADDED: `_load_ts` (silver load timestamp)
- CARRIED: `_batch`, `_run_id`
- Type cast: CT_CA_ID→BIGINT, CT_DTS→TIMESTAMP, CT_AMT→DECIMAL(12,2)
- CT_NAME stays STRING
- Dedup by (CT_CA_ID, CT_DTS, CT_AMT, CT_NAME) for re-run protection only

**Expected Count:** 1,204,943 rows

**Downstream:** gold.fact_cash_transactions → gold.fact_cash_balances (derived)

###### Author: Prajwol Regmi

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import (
    current_timestamp, col, row_number, lit, count, when
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    LongType, DecimalType, TimestampType
)
from pyspark.sql import Row
from datetime import datetime

# ─── CONFIG ──────────────────────────────────────────────────────────────
team_name = "team_lemma"
catalog_name = f"charles_schwab_retailbrokerage_dev_{team_name}"
bronze_schema = "bronze"
silver_schema = "silver"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {silver_schema}")

# Fully qualified table names
SOURCE_TABLE = f"{catalog_name}.{bronze_schema}.cashtransaction"
TARGET_TABLE = f"{catalog_name}.{silver_schema}.cash_transactions"

print(f"Catalog: {catalog_name}")
print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")
print(f"Mode: CREATE OR REPLACE (full rebuild with dedup)")

In [0]:
# ============================================================
# READ FROM BRONZE: bronze.cashtransaction
# ============================================================
# Source: CashTransaction.txt
#   B1: 1,203,664 rows (4 fields: CT_CA_ID, CT_DTS, CT_AMT, CT_NAME)
#   B2: 637 rows (6 fields: +CDC_FLAG, +CDC_DSN prefix, all I=Insert)
#   B3: 642 rows (6 fields: +CDC_FLAG, +CDC_DSN prefix, all I=Insert)
# Total Bronze: 1,204,943 rows (append-only, cumulative)
#
# Per MD: "Full rebuild from all Bronze" - read entire table
# CDC_FLAG is 'I' only (no updates or deletes)
# ============================================================

print("="*70)
print("SILVER.CASH_TRANSACTIONS: Bronze → Silver Processing")
print("="*70)
print("\n[Read] bronze.cashtransaction")
print("-"*70)

# Read ALL data from Bronze (full rebuild pattern)
bronze_cash = spark.table(SOURCE_TABLE)
source_count = bronze_cash.count()

print(f"  Total Bronze rows: {source_count} (expected: 1,204,943)")

# Profile by batch
print(f"\n  Batch distribution:")
bronze_cash.groupBy("_batch_id", "CDC_FLAG").count().orderBy("_batch_id").show()

# Profile data types before cast
print(f"  Source schema (ALL STRING from Bronze):")
for name, dtype in bronze_cash.dtypes:
    if name in ["CT_CA_ID", "CT_DTS", "CT_AMT", "CT_NAME"]:
        print(f"    {name:15s} {dtype} → needs type cast")

In [0]:
# ============================================================
# TYPE CAST + DEDUP
# ============================================================
# Per MD Silver Layer:
#   - Type cast: CT_CA_ID→BIGINT, CT_DTS→TIMESTAMP, CT_AMT→DECIMAL(12,2)
#   - CT_NAME stays STRING (no cast)
#   - Dedup by (CT_CA_ID, CT_DTS, CT_AMT, CT_NAME) for re-run protection
#     (same record ingested multiple times across reruns)
#   - All transactions are unique in normal flow (I=Insert only)
#
# Per MD Silver Audit Columns:
#   - DROPPED: _ingest_ts, _source_file, _source_name, CDC_FLAG, CDC_DSN
#   - ADDED: _load_ts
#   - CARRIED: _batch (from _batch_id), _run_id
# ============================================================

print("\n[Transform] Type Cast + Dedup")
print("-"*70)

# ── STEP 1: Type Cast Business Columns ─────────────────────────────
cash_typed = bronze_cash.select(
    col("CT_CA_ID").cast(LongType()).alias("CT_CA_ID"),
    col("CT_DTS").cast(TimestampType()).alias("CT_DTS"),
    col("CT_AMT").cast(DecimalType(12, 2)).alias("CT_AMT"),
    col("CT_NAME"),                                           # STRING - no cast
    # Audit columns to carry
    col("_batch_id"),
    col("_run_id"),
    col("_ingest_ts")  # Kept temporarily for dedup ordering, dropped after
)

print(f"  1. Type cast applied:")
print(f"       CT_CA_ID: STRING → BIGINT")
print(f"       CT_DTS:   STRING → TIMESTAMP")
print(f"       CT_AMT:   STRING → DECIMAL(12,2)")
print(f"       CT_NAME:  STRING (no cast)")

# Check for type cast failures (nulls introduced by bad data)
null_ca_id = cash_typed.filter(col("CT_CA_ID").isNull()).count()
null_dts = cash_typed.filter(col("CT_DTS").isNull()).count()
null_amt = cash_typed.filter(col("CT_AMT").isNull()).count()
print(f"\n  Type cast null check (should be 0):")
print(f"       CT_CA_ID nulls: {null_ca_id}")
print(f"       CT_DTS nulls:   {null_dts}")
print(f"       CT_AMT nulls:   {null_amt}")

# ── STEP 2: Dedup by composite key for re-run protection ─────────────
# ROW_NUMBER() PARTITION BY (CT_CA_ID, CT_DTS, CT_AMT, CT_NAME)
# ORDER BY _ingest_ts DESC (latest ingestion wins if duplicate)
dedup_window = Window.partitionBy(
    "CT_CA_ID", "CT_DTS", "CT_AMT", "CT_NAME"
).orderBy(col("_ingest_ts").desc())

cash_deduped = (
    cash_typed
    .withColumn("_row_num", row_number().over(dedup_window))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

dedup_count = cash_deduped.count()
rows_removed = source_count - dedup_count
print(f"\n  2. Dedup by (CT_CA_ID, CT_DTS, CT_AMT, CT_NAME):")
print(f"       Before: {source_count}")
print(f"       After:  {dedup_count}")
print(f"       Removed: {rows_removed} duplicate(s)")

# ── STEP 3: Apply Silver audit columns ─────────────────────────────
# Per MD: DROP _ingest_ts, _source_file; ADD _load_ts; CARRY _batch, _run_id
silver_cash_df = (
    cash_deduped
    .select(
        # Business columns
        "CT_CA_ID", "CT_DTS", "CT_AMT", "CT_NAME",
        # Audit columns (per MD spec)
        col("_batch_id").alias("_batch"),            # CARRIED (renamed)
        col("_run_id"),                              # CARRIED
    )
    .withColumn("_load_ts", current_timestamp())     # ADDED
)

final_count = silver_cash_df.count()
print(f"\n  3. Silver audit columns applied:")
print(f"       Dropped: _ingest_ts, _source_file, _source_name, CDC_FLAG, CDC_DSN")
print(f"       Added:   _load_ts")
print(f"       Carried: _batch, _run_id")
print(f"\n  Final DataFrame: {final_count} rows (expected: 1,204,943)")
print(f"  Schema:")
for name, dtype in silver_cash_df.dtypes:
    print(f"    {name:15s} {dtype}")

In [0]:
# ============================================================
# WRITE: CREATE OR REPLACE silver.cash_transactions
# ============================================================
# Per MD: Mode = CREATE OR REPLACE for tables rebuilt each run
# Pattern: Full rebuild from all Bronze (not incremental)
# Idempotent: re-running produces identical result
# ============================================================

print("\n[Write] CREATE OR REPLACE silver.cash_transactions")
print("-"*70)

# Full overwrite (CREATE OR REPLACE pattern)
silver_cash_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE)

# Validate target count
cash_target_count = spark.table(TARGET_TABLE).count()
cash_run_id = silver_cash_df.select("_run_id").first()[0]

expected_count = 1204943
match_status = "✓ PASS" if cash_target_count == expected_count else "✗ MISMATCH"

print(f"  Write mode: OVERWRITE (CREATE OR REPLACE)")
print(f"  Target count: {cash_target_count} (expected: {expected_count}) [{match_status}]")
print(f"  Carried run_id: {cash_run_id}")

In [0]:
# ============================================================
# DATA QUALITY VALIDATION
# ============================================================
# Verify silver.cash_transactions meets all quality expectations:
#   1. Row count matches expected (1,204,943)
#   2. Composite key uniqueness (CT_CA_ID, CT_DTS, CT_AMT, CT_NAME)
#   3. CT_CA_ID not null and > 0
#   4. CT_DTS not null (valid timestamps)
#   5. CT_AMT not null (valid amounts)
#   6. CT_NAME not null (transaction description present)
#   7. Audit column completeness
# ============================================================

print("\n[DQ] Data Quality Validation")
print("-"*70)

silver_cash = spark.table(TARGET_TABLE)
dq_results = []

# DQ-1: Row count validation
actual_count = silver_cash.count()
dq_1_pass = actual_count == 1204943
dq_results.append({"rule": "ROW_COUNT", "expected": "1204943", "actual": str(actual_count), "status": "PASS" if dq_1_pass else "FAIL"})
print(f"  DQ-1 Row Count: {actual_count} (expected 1,204,943) [{'PASS' if dq_1_pass else 'FAIL'}]")

# DQ-2: Composite key uniqueness (re-run protection check)
pk_dupes = (
    silver_cash
    .groupBy("CT_CA_ID", "CT_DTS", "CT_AMT", "CT_NAME")
    .count()
    .filter(col("count") > 1)
    .count()
)
dq_2_pass = pk_dupes == 0
dq_results.append({"rule": "COMPOSITE_KEY_UNIQUE", "expected": "0 dupes", "actual": str(pk_dupes), "status": "PASS" if dq_2_pass else "FAIL"})
print(f"  DQ-2 Composite Key Uniqueness: {pk_dupes} duplicates [{'PASS' if dq_2_pass else 'FAIL'}]")

# DQ-3: CT_CA_ID not null
null_ca_id = silver_cash.filter(col("CT_CA_ID").isNull()).count()
dq_3_pass = null_ca_id == 0
dq_results.append({"rule": "NOT_NULL_CT_CA_ID", "expected": "0 nulls", "actual": str(null_ca_id), "status": "PASS" if dq_3_pass else "FAIL"})
print(f"  DQ-3 CT_CA_ID Not Null: {null_ca_id} nulls [{'PASS' if dq_3_pass else 'FAIL'}]")

# DQ-4: CT_DTS not null (valid timestamps)
null_dts = silver_cash.filter(col("CT_DTS").isNull()).count()
dq_4_pass = null_dts == 0
dq_results.append({"rule": "NOT_NULL_CT_DTS", "expected": "0 nulls", "actual": str(null_dts), "status": "PASS" if dq_4_pass else "FAIL"})
print(f"  DQ-4 CT_DTS Not Null: {null_dts} nulls [{'PASS' if dq_4_pass else 'FAIL'}]")

# DQ-5: CT_AMT not null (valid amounts)
null_amt = silver_cash.filter(col("CT_AMT").isNull()).count()
dq_5_pass = null_amt == 0
dq_results.append({"rule": "NOT_NULL_CT_AMT", "expected": "0 nulls", "actual": str(null_amt), "status": "PASS" if dq_5_pass else "FAIL"})
print(f"  DQ-5 CT_AMT Not Null: {null_amt} nulls [{'PASS' if dq_5_pass else 'FAIL'}]")

# DQ-6: CT_NAME not null
null_name = silver_cash.filter(col("CT_NAME").isNull()).count()
dq_6_pass = null_name == 0
dq_results.append({"rule": "NOT_NULL_CT_NAME", "expected": "0 nulls", "actual": str(null_name), "status": "PASS" if dq_6_pass else "FAIL"})
print(f"  DQ-6 CT_NAME Not Null: {null_name} nulls [{'PASS' if dq_6_pass else 'FAIL'}]")

# DQ-7: Audit column completeness (_run_id, _batch, _load_ts not null)
null_audit = silver_cash.filter(
    col("_run_id").isNull() | col("_batch").isNull() | col("_load_ts").isNull()
).count()
dq_7_pass = null_audit == 0
dq_results.append({"rule": "AUDIT_COMPLETENESS", "expected": "0 nulls", "actual": str(null_audit), "status": "PASS" if dq_7_pass else "FAIL"})
print(f"  DQ-7 Audit Completeness: {null_audit} rows missing audit cols [{'PASS' if dq_7_pass else 'FAIL'}]")

# DQ-8: Data type validation (CT_AMT should have valid decimal values)
neg_check = silver_cash.filter(col("CT_AMT") < 0).count()
pos_check = silver_cash.filter(col("CT_AMT") > 0).count()
print(f"  DQ-8 Amount Distribution: {pos_check} positive, {neg_check} negative (both expected)")
dq_results.append({"rule": "AMT_DISTRIBUTION", "expected": "both +/-", "actual": f"+{pos_check}/-{neg_check}", "status": "PASS" if neg_check > 0 and pos_check > 0 else "FAIL"})

# Summary
total_pass = sum(1 for r in dq_results if r["status"] == "PASS")
total_rules = len(dq_results)
print(f"\n  DQ Summary: {total_pass}/{total_rules} rules passed")

# Log DQ results to operations layer
for dq in dq_results:
    failed_rows = 0 if dq["status"] == "PASS" else int(dq["actual"]) if dq["actual"].isdigit() else 1
    log_dq_result(
        spark=spark,
        run_id=cash_run_id,
        table_name="silver.cash_transactions",
        rule_name=dq["rule"],
        failed_rows=failed_rows,
        total_rows=actual_count
    )

In [0]:
# ============================================================
# OPERATIONS LOGGING: Pipeline Reconciliation + Audit Event
# ============================================================
# Per project standard (from 02_common_utils/operations):
#   1. log_pipeline_recon() - source vs target count comparison
#   2. log_audit_event() - write operation audit trail
#   3. log_domain_run_status() - domain completion status
# ============================================================

print("\n[Ops] Operations Logging")
print("-"*70)

# 1. Pipeline Reconciliation
log_pipeline_recon(
    spark=spark,
    run_id=cash_run_id,
    batch_id="Batch3",               # Final batch processed (cumulative full rebuild)
    domain="ACCOUNT",
    table_name="cash_transactions",
    source_layer="bronze",
    target_layer="silver",
    source_count=source_count,        # Total Bronze rows read
    target_count=cash_target_count    # Final Silver rows after dedup
)
print(f"  ✓ log_pipeline_recon: source={source_count} → target={cash_target_count}")

# 2. Audit Event
log_audit_event(
    spark=spark,
    run_id=cash_run_id,
    batch="Batch3",
    layer="silver",
    table_name="cash_transactions",
    operation="OVERWRITE",
    rows_affected=cash_target_count
)
print(f"  ✓ log_audit_event: operation=OVERWRITE, rows={cash_target_count}")

# 3. Domain Run Status
log_domain_run_status(
    spark=spark,
    run_id=cash_run_id,
    batch="Batch3",
    domain_name="ACCOUNT_SILVER_CASH_TRANSACTIONS",
    status="COMPLETED"
)
print(f"  ✓ log_domain_run_status: ACCOUNT_SILVER_CASH_TRANSACTIONS = COMPLETED")

print(f"\n  All operations logged successfully (run_id: {cash_run_id})")

In [0]:
# ============================================================
# VERIFICATION: silver.cash_transactions
# ============================================================

print("[Verify] silver.cash_transactions")
print("="*70)

silver_ct = spark.table(TARGET_TABLE)

# Count
print(f"  Total rows: {silver_ct.count()} (expected: 1,204,943)")

# Schema
print(f"\n  Schema:")
for c in silver_ct.dtypes:
    print(f"    {c[0]:15s} {c[1]}")

# Batch distribution
print(f"\n  Batch distribution:")
silver_ct.groupBy("_batch").count().orderBy("_batch").show()

# Amount statistics
print(f"  Amount statistics:")
silver_ct.selectExpr(
    "min(CT_AMT) as min_amt",
    "max(CT_AMT) as max_amt",
    "avg(CT_AMT) as avg_amt",
    "count(CASE WHEN CT_AMT > 0 THEN 1 END) as positive_txns",
    "count(CASE WHEN CT_AMT < 0 THEN 1 END) as negative_txns"
).show(truncate=False)

# Timestamp range
print(f"  Timestamp range:")
silver_ct.selectExpr(
    "min(CT_DTS) as earliest_txn",
    "max(CT_DTS) as latest_txn"
).show(truncate=False)

# Distinct accounts
distinct_accounts = silver_ct.select("CT_CA_ID").distinct().count()
print(f"  Distinct accounts (CT_CA_ID): {distinct_accounts}")

# Sample data
print(f"\n  Sample records (first 15):")
display(silver_ct.orderBy("CT_CA_ID", "CT_DTS").limit(15))